# Optional pseudo confidence generation

This notebook downloads saved ensemble model matrices from Hugging Face, fuses B7/B6/ConvNeXt scores, exports a pseudo-label confidence CSV, and optionally uploads the CSV back to Hugging Face.

In [ ]:
!pip install -q huggingface_hub

import os
import shutil
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, upload_file, login, create_repo

# ============================================================
# Hugging Face settings
# ============================================================

HF_MODEL_MATS_REPO_ID = "liu-peilin/happywhale_ensemble_b6_b7_conv_v1"
HF_MODEL_MATS_REPO_TYPE = "model"
HF_MODEL_MATS_DIR = "model_mats"

LABEL_CLASSES_REPO_ID = "liu-peilin/happywhale_b6_pseudo_1024_v1"
LABEL_CLASSES_REPO_TYPE = "model"
LABEL_CLASSES_HF_PATH = "metadata/label_classes.npy"

UPLOAD_CONFIDENCE_TO_HF = True

HF_OUTPUT_REPO_ID = "liu-peilin/happywhale_ensemble_v2xl_b6_b7_conv_v1"
HF_OUTPUT_REPO_TYPE = "model"
HF_OUTPUT_DIR = "pseudo_confidence"

LOCAL_ROOT = "/content/hf_pseudo_confidence"
LOCAL_MODEL_MATS_DIR = os.path.join(LOCAL_ROOT, "model_mats")
LOCAL_METADATA_DIR = os.path.join(LOCAL_ROOT, "metadata")
OUT_DIR = "/content/pseudo_confidence_output"

os.makedirs(LOCAL_MODEL_MATS_DIR, exist_ok=True)
os.makedirs(LOCAL_METADATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# HF token
# Download public files may not need token, but upload requires token
# ============================================================

def get_hf_token_required():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    if token:
        return token

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            return token
    except Exception:
        pass

    from getpass import getpass
    token = getpass("Paste Hugging Face token with write permission: ")
    os.environ["HF_TOKEN"] = token
    return token


HF_TOKEN = get_hf_token_required()
login(token=HF_TOKEN, add_to_git_credential=False)

create_repo(
    repo_id=HF_OUTPUT_REPO_ID,
    repo_type=HF_OUTPUT_REPO_TYPE,
    private=True,
    exist_ok=True,
    token=HF_TOKEN,
)

print("HF output repo ready:", HF_OUTPUT_REPO_ID)

# ============================================================
# Helper functions
# ============================================================

def download_hf_file_if_missing(repo_id, repo_type, hf_path, local_path):
    if os.path.exists(local_path):
        print("Local file exists, skip download:", local_path)
        return local_path

    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    print("Downloading from HF:")
    print(" repo:", repo_id)
    print(" file:", hf_path)
    print(" ->", local_path)

    downloaded = hf_hub_download(
        repo_id=repo_id,
        repo_type=repo_type,
        filename=hf_path,
        token=HF_TOKEN,
    )

    shutil.copy2(downloaded, local_path)
    return local_path


def download_model_mat_file(filename):
    hf_path = f"{HF_MODEL_MATS_DIR}/{filename}"
    local_path = os.path.join(LOCAL_MODEL_MATS_DIR, filename)

    return download_hf_file_if_missing(
        repo_id=HF_MODEL_MATS_REPO_ID,
        repo_type=HF_MODEL_MATS_REPO_TYPE,
        hf_path=hf_path,
        local_path=local_path,
    )


# ============================================================
# Target ensemble setting
# ============================================================

B7_W = 0.70
B6_W = 0.20
CONV_W = 0.10

THRESHOLD = 0.324167
NEW_RATIO = 0.200

TARGET_STEM = (
    "ensemble_b7pseudofb085_b6pseudofb080_convpseudo_fb085"
    "_knn050_logit025_proto025_species0.00"
    "_b7_070_b6_020_conv_010"
    "_newratio0.200_th0.324167"
)

# ============================================================
# Model mat filenames
# These filenames must exactly match files in HF model_mats/
# ============================================================

B7_MAT_FILE = "b7_b7pseudo_fb085_bf005_none010_knn050_logit025_proto025_species0.00_model_mat.npy"
B7_IMG_FILE = "b7_b7pseudo_fb085_bf005_none010_knn050_logit025_proto025_species0.00_test_images.npy"

B6_MAT_FILE = "b6_b6pseudo_fb080_bf010_none010_knn050_logit025_proto025_species0.00_model_mat.npy"
B6_IMG_FILE = "b6_b6pseudo_fb080_bf010_none010_knn050_logit025_proto025_species0.00_test_images.npy"

CONV_MAT_FILE = "conv_convpseudo_fb085_bf005_none010_knn050_logit025_proto025_species0.00_model_mat.npy"
CONV_IMG_FILE = "conv_convpseudo_fb085_bf005_none010_knn050_logit025_proto025_species0.00_test_images.npy"

# ============================================================
# Download label_classes.npy
# ============================================================

LABEL_CLASSES_PATH = os.path.join(LOCAL_METADATA_DIR, "label_classes.npy")

download_hf_file_if_missing(
    repo_id=LABEL_CLASSES_REPO_ID,
    repo_type=LABEL_CLASSES_REPO_TYPE,
    hf_path=LABEL_CLASSES_HF_PATH,
    local_path=LABEL_CLASSES_PATH,
)

# ============================================================
# Download model_mats and test_images from HF
# ============================================================

B7_MAT_PATH = download_model_mat_file(B7_MAT_FILE)
B6_MAT_PATH = download_model_mat_file(B6_MAT_FILE)
CONV_MAT_PATH = download_model_mat_file(CONV_MAT_FILE)

B7_IMG_PATH = download_model_mat_file(B7_IMG_FILE)
B6_IMG_PATH = download_model_mat_file(B6_IMG_FILE)
CONV_IMG_PATH = download_model_mat_file(CONV_IMG_FILE)

required_paths = [
    LABEL_CLASSES_PATH,
    B7_MAT_PATH,
    B6_MAT_PATH,
    CONV_MAT_PATH,
    B7_IMG_PATH,
    B6_IMG_PATH,
    CONV_IMG_PATH,
]

for p in required_paths:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")

print("All required files exist.")

# ============================================================
# Load label classes and test image order
# ============================================================

label_classes = np.load(LABEL_CLASSES_PATH, allow_pickle=True)
num_classes = len(label_classes)

b7_images = np.load(B7_IMG_PATH, allow_pickle=True)
b6_images = np.load(B6_IMG_PATH, allow_pickle=True)
conv_images = np.load(CONV_IMG_PATH, allow_pickle=True)

assert np.array_equal(b7_images, b6_images), "B7 and B6 test image order mismatch!"
assert np.array_equal(b7_images, conv_images), "B7 and Conv test image order mismatch!"

test_images = b7_images

print("num_classes:", num_classes)
print("test_images:", test_images.shape)

# ============================================================
# Load matrices with mmap
# ============================================================

b7_mat = np.load(B7_MAT_PATH, mmap_mode="r")
b6_mat = np.load(B6_MAT_PATH, mmap_mode="r")
conv_mat = np.load(CONV_MAT_PATH, mmap_mode="r")

print("b7_mat:", b7_mat.shape, b7_mat.dtype)
print("b6_mat:", b6_mat.shape, b6_mat.dtype)
print("conv_mat:", conv_mat.shape, conv_mat.dtype)

assert b7_mat.shape == b6_mat.shape == conv_mat.shape, "Matrix shape mismatch!"
assert b7_mat.shape[0] == len(test_images), "Matrix rows and test images mismatch!"
assert b7_mat.shape[1] == num_classes, "Matrix class dim and label_classes mismatch!"

# ============================================================
# Chunk-wise final ensemble + confidence extraction
# ============================================================

chunk_size = 256
topk = 5

rows = []

for start in tqdm(range(0, b7_mat.shape[0], chunk_size), desc="Generate confidence CSV"):
    end = min(start + chunk_size, b7_mat.shape[0])

    chunk = (
        B7_W * np.asarray(b7_mat[start:end], dtype=np.float32)
        + B6_W * np.asarray(b6_mat[start:end], dtype=np.float32)
        + CONV_W * np.asarray(conv_mat[start:end], dtype=np.float32)
    )

    top_idx = np.argpartition(-chunk, kth=topk - 1, axis=1)[:, :topk]
    top_scores = np.take_along_axis(chunk, top_idx, axis=1)

    order = np.argsort(-top_scores, axis=1)
    top_idx = np.take_along_axis(top_idx, order, axis=1)
    top_scores = np.take_along_axis(top_scores, order, axis=1)

    for bi in range(end - start):
        global_i = start + bi

        idxs = top_idx[bi]
        scores = top_scores[bi]

        top1_idx = int(idxs[0])
        top2_idx = int(idxs[1])

        top1_label = str(label_classes[top1_idx])
        top2_label = str(label_classes[top2_idx])

        top1_score = float(scores[0])
        top2_score = float(scores[1])
        margin = top1_score - top2_score

        is_new_by_threshold = top1_score < THRESHOLD
        pseudo_label = "" if is_new_by_threshold else top1_label

        top5_labels = [str(label_classes[int(x)]) for x in idxs]
        top5_scores = [float(x) for x in scores]

        rows.append({
            "image": str(test_images[global_i]),
            "top1_label": top1_label,
            "top1_pred": top1_label,
            "top1_score": top1_score,
            "top2_label": top2_label,
            "top2_pred": top2_label,
            "top2_score": top2_score,
            "margin": margin,
            "score_margin_top1_top2": margin,
            "new_threshold": THRESHOLD,
            "new_ratio": NEW_RATIO,
            "is_new_by_threshold": bool(is_new_by_threshold),
            "pseudo_label": pseudo_label,
            "top5_labels": " ".join(top5_labels),
            "top5_scores": " ".join([f"{s:.8f}" for s in top5_scores]),
        })

conf_df = pd.DataFrame(rows)

# ============================================================
# Save confidence CSV locally
# ============================================================

confidence_path = os.path.join(
    OUT_DIR,
    f"{TARGET_STEM}_confidence.csv",
)

conf_df.to_csv(confidence_path, index=False)

print("\nSaved confidence CSV:")
print(confidence_path)
print(conf_df.head())

# ============================================================
# Upload confidence CSV to HF
# ============================================================

hf_conf_path = f"{HF_OUTPUT_DIR}/{os.path.basename(confidence_path)}"

upload_file(
    path_or_fileobj=confidence_path,
    path_in_repo=hf_conf_path,
    repo_id=HF_OUTPUT_REPO_ID,
    repo_type=HF_OUTPUT_REPO_TYPE,
    token=HF_TOKEN,
)

print("Uploaded confidence CSV to HF:", hf_conf_path)

# ============================================================
# Quick pseudo count preview
# ============================================================

print("\nPseudo label count preview:")

for top1_th, margin_th in [
    (0.80, 0.15),
    (0.75, 0.12),
    (0.70, 0.10),
]:
    pseudo_df = conf_df[
        (conf_df["is_new_by_threshold"] == False)
        & (conf_df["top1_score"] >= top1_th)
        & (conf_df["margin"] >= margin_th)
    ]

    print(
        f"top1_score >= {top1_th}, margin >= {margin_th}: "
        f"{len(pseudo_df)} pseudo images"
    )

print("\nDone.")